# LYT-Net Ablation — Local Mac runner

Runs the ablation study directly on this Mac using the existing TF venv. **No uploads, no Drive, no Colab.**

## Before running

**Pick the right Python kernel in VS Code** — this is what caused the `ModuleNotFoundError: No module named 'tensorflow'` you saw earlier.

1. Top-right of the notebook → click the kernel selector (says `Python 3.x.x` or `Select Kernel`).
2. Choose **Python Environments…** → look for the entry whose path is:
   `LYT-Net-main/TensorFlow/venv/bin/python`
   (Python 3.11.15). That's the venv that already has TensorFlow 2.16.2, tensorflow-metal, PyTorch, lpips, and everything else.
3. If you don't see it, click **"Enter interpreter path…"** and paste:
   `/Users/mba/Desktop/YZU/GenAI/LYT/LYT-Net-main/TensorFlow/venv/bin/python`

Then run Cell 1 below to verify.

## Cell 1 — Verify kernel and TensorFlow / Metal GPU

In [ ]:
import sys, os
print('Python:', sys.executable)
print('Version:', sys.version.split()[0])

expected = 'LYT-Net-main/TensorFlow/venv/bin/python'
if expected not in sys.executable:
    print(f'\nWARNING: kernel is not the project venv. Expected path ending in {expected}.')
    print('Fix: top-right kernel selector → Enter interpreter path → paste the venv python path from the markdown above.')

try:
    import tensorflow as tf
    print('\nTensorFlow:', tf.__version__)
    gpus = tf.config.list_physical_devices('GPU')
    print('GPUs:', gpus or 'CPU only (tensorflow-metal not loaded)')
    if not gpus:
        print('  → Training will fall back to CPU and be much slower.')
    import torch, lpips
    print('PyTorch:', torch.__version__, '   LPIPS imported OK')
except ModuleNotFoundError as e:
    print(f'\nMISSING: {e.name}. You are on the wrong kernel — see the markdown above.')
    raise

## Cell 2 — cd into TensorFlow/ and verify paths

Datasets are already on disk in the right place. This cell just confirms it.

Required structure (already true in your repo):
```
LYT-Net-main/TensorFlow/data/LOLv1/Train/{input,target}
LYT-Net-main/TensorFlow/data/LOLv1/Test/{input,target}
LYT-Net-main/TensorFlow/data/LOLv2/Real_captured/Test/{Low,Normal}
LYT-Net-main/TensorFlow/pretrained_weights/LOLv1.h5
LYT-Net-main/dataset/LoLI-Street Dataset/Val/{low,high}
```

In [ ]:
import os

REPO_ROOT = '/Users/mba/Desktop/YZU/GenAI/LYT/LYT-Net-main'
TF_DIR    = f'{REPO_ROOT}/TensorFlow'
os.chdir(TF_DIR)
print('Working dir:', os.getcwd())

def check(label, path, kind='dir'):
    ok = os.path.isdir(path) if kind == 'dir' else os.path.isfile(path)
    n  = len(os.listdir(path)) if ok and kind == 'dir' else (1 if ok else 0)
    print(f"  [{'OK ' if ok else 'MISS'}] {label:<28} {path}   ({n})")
    return ok

ok = True
ok &= check('LOLv1 Train/input',   'data/LOLv1/Train/input')
ok &= check('LOLv1 Train/target',  'data/LOLv1/Train/target')
ok &= check('LOLv1 Test/input',    'data/LOLv1/Test/input')
ok &= check('LOLv1 Test/target',   'data/LOLv1/Test/target')
ok &= check('LOLv2-Real Test/Low', 'data/LOLv2/Real_captured/Test/Low')
ok &= check('LOLv2-Real Test/Norm','data/LOLv2/Real_captured/Test/Normal')
ok &= check('LoLI Val/low',        '../dataset/LoLI-Street Dataset/Val/low')
ok &= check('LoLI Val/high',       '../dataset/LoLI-Street Dataset/Val/high')
ok &= check('Baseline weights',    'pretrained_weights/LOLv1.h5', 'file')
assert ok, 'Fix the missing items above before training.'
print('\nAll paths present.')

## Cell 3 — Set epoch budget

`train_ablation.py` epoch count is patched by this cell. This patches it for a realistic Mac run.

Speed reality on Apple Silicon GPU (Metal, batch_size=1, 485 train images per epoch):

| `EPOCHS` | Per variant | All 7 |
|---|---|---|
| 1000 (paper) | ~10–15 h | ~3 days |
| 300 | ~3–5 h | ~24 h |
| 100 | ~1–1.5 h | ~7–10 h |
| 30 (smoke test) | ~20 min | ~2.5 h |

A T4 in Colab is ~3× faster than M-series Metal. If you need paper-grade numbers, Colab is still the right venue.

In [ ]:
EPOCHS = 20   # ← set this. 1000 = paper, 300 = solid, 100 = decent, 30 = smoke test

import re
path = 'scripts/train_ablation.py'
src  = open(path).read()
src2 = re.sub(r'^TOTAL_EPOCHS\s*=\s*\d+', f'TOTAL_EPOCHS = {EPOCHS}', src, count=1, flags=re.M)
open(path,'w').write(src2)
print(f'TOTAL_EPOCHS set to {EPOCHS}')

## Cell 4 — Smoke test (5 epochs of D1 only)

**Run this first** before committing hours to full training. It verifies the model builds, the data loads, and a few epochs complete without crashing.

Takes ~3–5 min on Metal. If this cell fails, the full run will fail too — and I can debug from the traceback.

In [ ]:
import re, shutil

# Temporarily set epochs to 5 just for the smoke test
path = 'scripts/train_ablation.py'
src_orig = open(path).read()
src_5    = re.sub(r'^TOTAL_EPOCHS\s*=\s*\d+', 'TOTAL_EPOCHS = 5', src_orig, count=1, flags=re.M)
open(path,'w').write(src_5)

# Wipe any prior smoke-test checkpoints
import os
if os.path.isdir('experiments/ablation_d1'):
    shutil.rmtree('experiments/ablation_d1')

try:
    !{sys.executable} scripts/train_ablation.py --variant d1
finally:
    # Restore the user's chosen EPOCHS for the real run
    open(path,'w').write(src_orig)
    print(f'\nRestored TOTAL_EPOCHS to {EPOCHS}')

## Cell 5 — Train all 7 variants

This is the long one. Sequential: d1 → d2 → d3 → d1d2 → d1d3 → d2d3 → d1d2d3. Each variant writes its best checkpoint to `experiments/ablation_<variant>/net_<variant>_psnr_…_epoch_….h5`.

**Keep your Mac awake** (System Settings → Lock Screen → Turn display off after: 1 hour at least, and disable sleep on battery). Closing the lid will pause training.

In [ ]:
!{sys.executable} scripts/train_ablation.py --variant all

## Cell 5b — (Alternative) Train one variant at a time

Use if Cell 5 crashed partway and you want to resume from where it died, instead of retraining variants that finished cleanly.

In [ ]:
# !python scripts/train_ablation.py --variant d1
# !python scripts/train_ablation.py --variant d2
# !python scripts/train_ablation.py --variant d3
# !python scripts/train_ablation.py --variant d1d2
# !python scripts/train_ablation.py --variant d1d3
# !python scripts/train_ablation.py --variant d2d3
# !python scripts/train_ablation.py --variant d1d2d3

## Cell 6 — Evaluate every variant on the 3 datasets

Walks `experiments/ablation_*/` and picks the best checkpoint per variant by filename PSNR, then evaluates each on LOL-v1 / LOL-v2-Real / LOLI-Street with PSNR / SSIM / LPIPS. Writes `experiments/ablation_results.csv`.

In [ ]:
import glob, os

def best_h5(d):
    cs = glob.glob(os.path.join(d,'*.h5'))
    def psnr(p):
        try: return float(os.path.basename(p).split('psnr_')[1].split('_')[0])
        except: return 0.0
    return max(cs, key=psnr) if cs else None

full = best_h5('experiments/ablation_d1d2d3') or best_h5('experiments/LOLv1')
print('Full-model weights:', full)

!{sys.executable} scripts/eval_ablation.py --auto \
    --baseline_weights pretrained_weights/LOLv1.h5 \
    --full_weights "$full"

## Cell 7 — Pretty-print the filled table

In [ ]:
import pandas as pd

df = pd.read_csv('experiments/ablation_results.csv')

ORDER = ['Baseline (no mod.)', '+D1 only', '+D2 only', '+D3 only',
         '+D1+D2', '+D1+D3', '+D2+D3', 'D1+D2+D3 (Ours)']
df['variant'] = pd.Categorical(df['variant'], categories=ORDER, ordered=True)

pivot = df.pivot_table(index='variant', columns='dataset',
                       values=['psnr','ssim','lpips'], aggfunc='first')
pivot.columns = [f'{m}_{ds}' for m, ds in pivot.columns]
DS = ['LOLv1','LOLv2_Real','LOLI-Street']
cols = [f'{m}_{ds}' for ds in DS for m in ['psnr','ssim','lpips']]
pivot = pivot.reindex(index=ORDER, columns=cols)

print(pivot.round(4).to_string())
pivot.round(4).to_csv('experiments/ablation_results_pivot.csv')
print('\nSaved → experiments/ablation_results_pivot.csv')

## Cell 8 — Emit filled LaTeX table

Writes `../ablation_table_filled.tex` (next to your existing `ablation_section_final.tex`). Bold = best in column (max for PSNR/SSIM, min for LPIPS).

In [ ]:
import pandas as pd
df = pd.read_csv('experiments/ablation_results.csv')

ROWS = [('Baseline',           'Baseline (no mod.)'),
        ('+D1',                '+D1 only'),
        ('+D2',                '+D2 only'),
        ('+D3',                '+D3 only'),
        ('+D1+D2',             '+D1+D2'),
        ('+D1+D3',             '+D1+D3'),
        ('+D2+D3',             '+D2+D3'),
        ('D1+D2+D3 (Ours)',    'D1+D2+D3 (Ours)')]
DS_ORDER = ['LOLv1','LOLv2_Real','LOLI-Street']

table = {(r.variant, r.dataset): (r.psnr, r.ssim, r.lpips) for r in df.itertuples()}

best = {}
for ds in DS_ORDER:
    psnrs = [table[(v,ds)][0] for _,v in ROWS if (v,ds) in table]
    ssims = [table[(v,ds)][1] for _,v in ROWS if (v,ds) in table]
    lps   = [table[(v,ds)][2] for _,v in ROWS if (v,ds) in table]
    if psnrs: best[(ds,'psnr')]  = max(psnrs)
    if ssims: best[(ds,'ssim')]  = max(ssims)
    if lps:   best[(ds,'lpips')] = min(lps)

def fmt(val, key, prec):
    s = f'{val:.{prec}f}'
    return f'\\textbf{{{s}}}' if abs(val - best[key]) < 1e-4 else s

lines = [
    r'\begin{table}[t]',
    r'  \centering',
    r'  \caption{Ablation study. All variants trained on LOL-v1, evaluated without',
    r'  fine-tuning. $\uparrow$: higher is better; $\downarrow$: lower is better.',
    r'  \textbf{Bold}: best in column.}',
    r'  \label{tab:ablation}',
    r'  \setlength{\tabcolsep}{3.5pt}',
    r'  \begin{tabular}{lccccccccc}',
    r'    \toprule',
    r'    & \multicolumn{3}{c}{\textbf{LOL-v1}}',
    r'    & \multicolumn{3}{c}{\textbf{LOL-v2-Real}}',
    r'    & \multicolumn{3}{c}{\textbf{LOLI-Street}} \\',
    r'    \cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-10}',
    r'    \textbf{Variant}',
    r'      & PSNR$\uparrow$ & SSIM$\uparrow$ & LPIPS$\downarrow$',
    r'      & PSNR$\uparrow$ & SSIM$\uparrow$ & LPIPS$\downarrow$',
    r'      & PSNR$\uparrow$ & SSIM$\uparrow$ & LPIPS$\downarrow$ \\',
    r'    \midrule',
]

for label, key in ROWS:
    cells = []
    for ds in DS_ORDER:
        if (key, ds) not in table:
            cells += ['---','---','---']; continue
        p,s,l = table[(key, ds)]
        cells += [fmt(p,(ds,'psnr'),2), fmt(s,(ds,'ssim'),3), fmt(l,(ds,'lpips'),3)]
    if label == 'D1+D2+D3 (Ours)':
        lines.append(r'    \midrule')
    lines.append(f"    {label:<20} & " + ' & '.join(cells) + r' \\')

lines += [r'    \bottomrule', r'  \end{tabular}', r'\end{table}']
out = '\n'.join(lines) + '\n'
open('../ablation_table_filled.tex','w').write(out)
print(out)
print('\nWrote: ../ablation_table_filled.tex')